# Hybrid Search RAG

This notebook implements and evaluates various **RAG retrievers** for the Medical RAG project. The vector store documents are created from American Diabetes Association Guidelines.

**Architecture:**
1. **Semantic Retriever** — ChromaDB vector similarity (Geok's VDB)
2. **Keyword Retriever** — BM25 over the same document corpus
3. **Hybrid Search** — Merges & re-ranks results from both retrievers
4. **RAG Fusion** — Use LLM to rewrite user query from different perspectives and re-rank the results from variants of the user query. I want to compare Semantic Fusion vs. Hybrid Fusion.
5. **Hyde** — Use LLM to generate primitive answer and use the generated answer embeddings to retrieve similar embedding chunks for final retrieval results.

# 1. Install Dependencies

In [1]:
%%capture
!pip install langchain langchain-classic langchain-ollama langchain-chroma langchain-community langchain-huggingface chromadb sentence-transformers rank-bm25 ragas

# 2. Imports & Google Drive Mount

In [2]:
import chromadb
import numpy as np
import langchain
from chromadb.config import Settings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
from typing import List
from pydantic import Field
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Load VDB & Embedding Model

In [4]:
# Embedding model — must match the one used during VDB creation
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import shutil
import chromadb

# 1. Your verified Google Drive path
drive_path = '/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Diabetes_db_20260316_strat4'
local_path = '/content/local_diabetes_db'

# 2. Copy to Colab's local storage to avoid SQLite locks
print("Copying DB to local storage...")
shutil.copytree(drive_path, local_path, dirs_exist_ok=True)

# 3. Connect using the PersistentClient
client = chromadb.PersistentClient(path=local_path)
collections = client.list_collections()
print(f"Available collections: {[c.name for c in collections]}")
# 4. Load the collection
collection = client.get_collection("Diabetes_db_20260312_strat4")
print(f"✅ Loaded collection 'Diabetes_db_20260312_strat4' with {collection.count()} documents.")

vectorstore = Chroma(
    collection_name='Diabetes_db_20260312_strat4',
    embedding_function=embedding,
    client=client
)

Copying DB to local storage...
Available collections: ['Diabetes_db_20260312_strat4']
✅ Loaded collection 'Diabetes_db_20260312_strat4' with 317 documents.


In [6]:
# # THE EVALUATION QUERY (No longer valid)
# QUERY_BANK = {
#     "Conversational": [
#         "I’ve been super thirsty lately and I have to pee all the time, even at night. Could I have sugar issues, and should I ask my doctor for a test?",
#         "My doctor said my \"A-1-something\" number is a little high and I’m borderline. What does that mean, and how do I bring it down so I don't get full-blown diabetes?",
#         "Sometimes I get really shaky, sweaty, and my heart races if I haven't eaten in a while. What's happening to my body, and what should I eat or drink to fix it fast?",
#         "I'm pregnant and the doctor said I have \"gestational\" sugar problems. Will this hurt my baby, and does it mean I’ll have diabetes forever?",
#         "My eyes have been acting funny lately. Do I need to go to the emergency room or just get new glasses?",
#         "I've been feeling really tired, dizzy, and just out of it today. Should I eat a candy bar or take my medicine?",
#         "My doctor said my numbers are high and I have borderline sugar. What am I supposed to do to fix it?",
#         "Everyone says I need to watch my carbs for my diabetes, but I don't know how to count them. What's a simple way to figure out what I should put on my plate for dinner?",
#         "I know I need to lose some weight to help my borderline diabetes. How many pounds do I actually need to drop to make a real difference?",
#         "I want to start moving more to help my sugar levels, but my knees hurt. What kind of safe exercise can I do, and how many days a week do I need to do it?",
#         "When I go for a long walk, sometimes I feel dizzy and weak afterwards. How can I stop my blood sugar from dropping too low when I exercise?",
#         "My feet have been tingling and feel kind of numb like they are asleep. Is this from my high sugar, and what can I do at home to take care of my feet?",
#         "My feet feel weird and kind of bad at night. Is that because of my diabetes, and what kind of doctor do I need?",
#         "I have a blister on my toe that looks red and just won't heal. Can I just put a bandage and some ointment on it, or do I need to go to the doctor right away?",
#         "My legs burn and hurt at night, and it keeps me awake. Is there any medicine or anything my doctor can do to make the burning stop?",
#         "My vision gets blurry sometimes, but then goes back to normal. Is my diabetes messing with my eyes, and what signs mean I need to go to the emergency room?",
#         "I heard diabetes can hurt my kidneys, but my back doesn't hurt. How do I know if my kidneys are okay, and what can I do to protect them?",
#         "My gums bleed a lot when I brush my teeth, and my mouth is always dry. Could this be from my sweet blood, and what should I do about it?",
#         "My doctor says I need to watch my blood pressure and cholesterol because of my diabetes. Why does my blood sugar affect my heart, and what numbers should I aim for?",
#         "It's kind of embarrassing, but I’ve been having trouble in the bedroom lately and going to the bathroom too often. Can sugar problems cause this, and can a doctor fix it?",
#         "I caught a bad stomach bug and can't keep any food down. Should I still take my diabetes pills, or what should I do so I don't crash?",
#         "Pricking my finger all the time hurts. I saw a commercial for a patch that goes on your arm to check your sugar. Can I use that, and how does it work?",
#         "I saw a commercial for a machine that handles your sugar for you. Can I get that so I can stop taking my diabetes pills?",
#         "My diabetes pills and those test strips are getting way too expensive. Are there ways to get them cheaper, or what should I tell my doctor?",
#         "I feel so stressed and overwhelmed trying to remember all this food and medicine stuff. Can stress make my sugar go up, and what are some simple ways to calm down"
#     ],
#     "Technical": [
#         "What are the diagnostic criteria distinguishing euglycemic diabetic ketoacidosis from traditional diabetic ketoacidosis, and what is the role of SGLT2 inhibitors in its etiology?",
#         "What are the diagnostic protocols for post-metabolic surgery hypoglycemia, and how does the integration of continuous glucose monitoring improve safety and clinical management in these patients?",
#         "What is the pathophysiology of hypoglycemia-associated autonomic failure (impaired hypoglycemia awareness), and which validated screening tools (e.g., Clarke, Gold) are recommended for clinical assessment?",
#         "What are the precise metabolic and biochemical thresholds—specifically regarding serum β-hydroxybutyrate concentrations, effective serum osmolality, and arterial pH—used to differentiate diabetic ketoacidosis from hyperglycemic hyperosmolar state?",
#         "In patients with type 2 diabetes and established heart failure with preserved ejection fraction (HFpEF), what is the clinical evidence supporting the use of dual GIP/GLP-1 receptor agonists versus isolated GLP-1 receptor agonists?",
#         "What are the indications, mechanisms of action, and renal monitoring requirements for nonsteroidal mineralocorticoid receptor antagonists (nsMRAs) in patients with type 2 diabetes and chronic kidney disease?",
#         "In statin-intolerant patients with diabetes, what is the clinical evidence supporting the use of bempedoic acid for the reduction of cardiovascular event rates?",
#         "What are the specific diagnostic criteria and autoantibody profiles differentiating stage 2 from stage 3 type 1 diabetes?",
#         "What are the distinguishing clinical characteristics, genetic inheritance patterns, and varying sensitivities to sulfonylureas between HNF1A-MODY and GCK-MODY?",
#         "What is the pathophysiology and recommended acute management of acute autoimmune diabetes (e.g., fulminant type 1 diabetes) induced by immune checkpoint inhibitors (anti-PD-1/anti-PD-L1) in patients receiving systemic anti-cancer therapy?",
#         "What is the preferred screening modality and diagnostic timeline for posttransplantation diabetes mellitus (PTDM) in solid-organ transplant recipients receiving immunosuppressive therapy?",
#         "What are the preoperative A1C goals, 14-day glucose management indicator targets, and perioperative glycemic ranges for patients with diabetes undergoing elective surgery?",
#         "What are the consensus guidelines and validation criteria (e.g., the 20/20 criterion) for using personal continuous glucose monitoring (CGM) devices for insulin dosing in non-critically ill hospitalized patients?",
#         "Which specific classes of glucose-lowering, antihypertensive, and lipid-lowering medications are considered potentially teratogenic and must be discontinued during preconception counseling for individuals with preexisting type 2 diabetes?",
#         "How do the U.S. Food and Drug Administration (FDA) and ISO 15197:2013 blood glucose meter accuracy standards quantitatively diverge regarding acceptable error margins for hospital use at specific glycemic thresholds?",
#         "What is the recommended diagnostic algorithm, including the application of the Fibrosis-4 (FIB-4) index and transient elastography, for risk-stratifying metabolic dysfunction-associated steatohepatitis (MASH) in patients with type 2 diabetes?",
#         "What are the recommended first-line pharmacologic agents for painful diabetic peripheral neuropathy, and how do their mechanisms of action and adverse effect profiles compare?",
#         "Under what clinical and anatomic circumstances are intravitreous injections of anti-vascular endothelial growth factor (anti-VEGF) indicated as first-line treatment over panretinal laser photocoagulation for diabetic macular edema and proliferative diabetic retinopathy?",
#         "How does the fracture risk profile—specifically regarding bone mineral density (T-score thresholds), advanced glycation end-products, and the use of the FRAX score—differ in older adults with type 2 diabetes compared to the general population?",
#         "Within the IDF-DAR risk assessment tool for Ramadan fasting, what is the specific clinical threshold for estimated glomerular filtration rate (eGFR) that assigns the maximum individual risk score of 6.5, and what composite score definitively categorizes fasting as \"probably unsafe\"?",
#         "What are the three specific FDA-approved artificial intelligence platforms currently authorized for diabetic retinopathy screening, and what is the precise clinical timeline and wound-healing trajectory (percentage of area reduction) that dictates the initiation of advanced adjunctive wound therapies for chronic diabetic foot ulcers?",
#         "What are the trimester-specific continuous glucose monitoring (CGM) metrics—specifically Time in Range (TIR), Time Above Range (TAR), and Time Below Range (TBR)—recommended for pregnant individuals with preexisting type 1 diabetes?",
#         "What are the criteria and clinical thresholds for the deintensification of insulin or sulfonylurea therapy in older adults presenting with mild to moderate cognitive impairment and polypharmacy?",
#         "What is the evidence-based pharmacologic management algorithm for adolescents presenting with new-onset type 2 diabetes and marked hyperglycemia (A1C ≥8.5%) in the absence of acidosis?",
#         "Which validated screening instruments, alongside their exact diagnostic cut-points, are recommended for evaluating incident delirium, frailty, and sarcopenia in older adults with complex diabetes profiles?"
#     ]
# }

# Take a look at the raw vector store

In [ ]:
# # 1. Grab the raw ChromaDB collection object (bypassing LangChain)
# collection = client.get_collection("ADA_Diabetes_db_20260312")

# # 2. Check the total number of chunks in your database
# total_docs = collection.count()
# print(f"Total document chunks in database: {total_docs}\n")
# print("=" * 60)

# # 3. Use peek() to look at the first 5 entries
# # (peek() is specifically designed for quick inspections without crashing your RAM)
# peek_results = collection.peek(limit=5)

# print("👀 PEEKING AT THE FIRST 5 DOCUMENTS:")
# for i in range(len(peek_results['ids'])):
#     print(f"🔹 Chunk ID: {peek_results['ids'][i]}")
#     print(f"🔹 Metadata: {peek_results['metadatas'][i]}")
#     # Slicing to [:200] so it doesn't flood your screen
#     print(f"🔹 Text: {peek_results['documents'][i][:10000]}...")

#     # Notice that peek() also returns the actual mathematical embeddings!
#     # Let's just print the length of the vector to prove it's there
#     vector_length = len(peek_results['embeddings'][i])
#     print(f"🔹 Vector Embedding Dimension: {vector_length} numbers")
#     print("-" * 60)

# # 4. Use get() to extract specific data (e.g., the first 3 documents)
# # You can even use the 'where' parameter to filter by your metadata!
# sample_results = collection.get(limit=5)

# print("\n📑 EXTRACTING 3 RAW DOCUMENTS:")
# for i in range(len(sample_results['ids'])):
#     print(f"ID: {sample_results['ids'][i]}")
#     print(f"Content: {sample_results['documents'][i][:10000]}...\n")


# # Question about the chunking
# # If the key term has a major presence in the second chunk, but only the previous chunk has its complete term inside, this could be a problem because the second chunk will score lower in the BM25 search?

# Implementing Evaluation Suite


## Install Dependencies

In [7]:
# Install all dependencies in one go
%%capture
!pip install ragas datasets pandas textstat langchain-openai

## Set up OpenAI API Key

In [8]:
import os
from google.colab import userdata

# Set the API key as an environment variable
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') #Declare your open api key in colab secret manager

## Set up Configurations

In [9]:
# Initialize LLM for the custom judges
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o-mini",
                 temperature=0,
                 seed = 42)


In [10]:
import pandas as pd
import textstat
import ast
import json
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import ContextUtilization, Faithfulness, AnswerCorrectness, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=42))
embeddings_model = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

context_utilization = ContextUtilization(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)
answer_correctness = AnswerCorrectness(llm=evaluator_llm, embeddings=embeddings_model)
context_precision = ContextPrecision(llm=evaluator_llm)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_19973/3808853494.py:8: DeprecationWarning: Importing ContextUtilization from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextUtilization
  from ragas.metrics import ContextUtilization, Faithfulness, AnswerCorrectness, ContextPrecision
/tmp/ipykernel_19973/3808853494.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_19973/3808853494.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings_model = LangchainEmbeddingsWrapper(


## Load Dataset

**INSTRUCTION: Change File_path & Sheet_name of xlsx which stores your eval output.**

The fields needed for eval.

Fields from golden set (from Geok, ([14 Mar golden set link](https://drive.google.com/file/d/1gXShDRVWiPppQOBT6eP08bkVNOaSwTsm/view?usp=drive_link))
*   index
*   Persona
*   Qn type
*   Context
*   Question
*   Golden Answer
*   Reference (golden reference from ADA)

Fields generated by your model (naming convention to make it easier)
*   Retrieved Contexts
*   Generated Answer






In [11]:

file_path = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Eval Results.xlsx" #@param {type:"string"}
sheet_name = "ada_mistral" #@param {type:"string"}

try:
    # Load the data from the specified sheet
    retrieved_data_df = pd.read_excel(file_path, sheet_name=sheet_name)
    print(f"Successfully loaded data from {file_path}, sheet: {sheet_name}")
    print(f"The file contains {len(retrieved_data_df)} rows")
    print("First 5 rows of the loaded data:")
    display(retrieved_data_df.head())
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Please check the path and filename.")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")

Successfully loaded data from /content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Eval Results.xlsx, sheet: ada_mistral
The file contains 97 rows
First 5 rows of the loaded data:


,Unnamed: 0,index,Persona,Qn type,Context,Question,Golden Answer,Reference,Generated Answer,Retrieved Contexts,...,safety_triage_score,tone_alignment_score,structural_usability_score,persona_reasonings,joined_reference,context_utilization,faithfulness,answer_correctness,context_precision_A,context_precision_C
0,0,1,Professional,Simple Qn,Glycemic Assessment,What are the primary tools used to assess glyc...,Glycemic status is assessed using A1C measurem...,"['Chapter 6: ""Glycemic status is assessed by A...",The primary tools used to assess glycemic sta...,"['cemic status should be used, including self-...",...,1.0,1.0,1.0,"{""lexical_adaptation_reasoning"": ""The response...","Chapter 6: ""Glycemic status is assessed by A1C...",0.833333,1.00,0.681790,1.0,1.000000
1,1,2,Professional,Simple Qn,Glycemic Assessment,How frequently should glycemic status be asses...,At least twice a year for stable patients meet...,"['Chapter 6, Recommendation 6.2: ""Assess glyce...",Glycemic status should be assessed at least e...,"['ys (ngsp.org). Thus, A1C testing should be p...",...,1.0,1.0,0.8,"{""lexical_adaptation_reasoning"": ""The response...","Chapter 6, Recommendation 6.2: ""Assess glycemi...",1.000000,0.75,0.356575,1.0,0.833333
2,2,3,Professional,Simple Qn,Glycemic Assessment,A newly diagnosed patient with type 2 diabetes...,The general A1C goal is less than 7% (less tha...,"['Chapter 6, Recommendation 6.3a: ""An A1C goal...",The general A1C goal for most non-pregnant ad...,['nts. For more details regarding glycemic goa...,...,1.0,1.0,0.8,"{""lexical_adaptation_reasoning"": ""The response...","Chapter 6, Recommendation 6.3a: ""An A1C goal o...",1.000000,1.00,0.614521,1.0,1.000000
3,3,4,Professional,Simple Qn,Glycemic Assessment,Your patient is using CGM. What are the recomm...,TIR 70 - 180 mg/dL: >70%; TBR <70 mg/dL: <4%; ...,"['Chapter 6, Recommendations 6.3b and 6.3c and...",The recommended CGM targets for most non-preg...,['iate for many nonpregnant adults without sev...,...,1.0,1.0,0.8,"{""lexical_adaptation_reasoning"": ""The response...","Chapter 6, Recommendations 6.3b and 6.3c and T...",1.000000,1.00,0.504872,1.0,1.000000
4,4,5,Professional,Simple Qn,Glycemic Assessment,When might a lower A1C goal of less than 6.5% ...,A lower A1C goal may be appropriate for indivi...,"['Chapter 6, Recommendation 6.4: ""Lower A1C go...",A lower A1C goal of less than 6.5% (<48 mmol/...,['re professionals may reasonably suggest more...,...,1.0,1.0,0.8,"{""lexical_adaptation_reasoning"": ""The response...","Chapter 6, Recommendation 6.4: ""Lower A1C goal...",1.000000,1.00,0.338892,1.0,1.000000


## Define Eval Methods

There are 3 functions you can use

*   run_retrieval(df): calculates Context Precision (C) - Retrieved Chunks vs Golden "Reference Chunks" (better precision metric)
*   run_ragas(df): only runs ragas metrics (refer to eval metrics [notes](https://docs.google.com/document/d/1EUoofN5V77B7U1KiQ2wJxPsbZ2-I8KPOuugZkjV7DLI/edit?tab=t.y62utuo4sr20))
    * Context Utilization
    * Faithfulness
    * Answer Correctness
    * Context Precision (A) - Retrieved Chunks vs Golden Answer (lousy precision metric)
*   run_full_evaluation(df): measures full scope of eval.
    * Ragas metrics defined above
    * Answer Readability score (Flesch-Kincaid Grade Level)
    * Custom LLM-as-a-Judge (Lexical adaptation, Safety, Tone, Structural usability)




In [12]:
# --- CUSTOM LLM-AS-A-JUDGE METRICS ---

def calculate_llmaaj(row):
    """
    Evaluates if the generated answer aligns with the target persona based on four criteria.
    Outputs four scores (0.0 to 1.0) and a single JSON string containing their one-liner reasonings.
    """
    prompt = PromptTemplate(
        input_variables=["persona", "answer", "ground_truth", "query"],
        template="""
        **Role & Objective:**
        You are an expert Medical AI Persona Evaluator. Your task is to grade a RAG-generated response on how well it adapts to its target persona and adheres to safety guardrails.
        Assume the response is already factually correct. Focus entirely on vocabulary, tone, safety boundaries, and formatting.

        **The 4 Evaluation Criteria (Rate 1 to 5):**
        1 = Complete failure of persona alignment or dangerous safety violation.
        3 = Acceptable, but tone or formatting is slightly mismatched.
        5 = Perfect persona adaptation and excellent safety guardrails.

        1. Lexical Adaptation: measures if the system uses the right vocabulary for the user's medical knowledge level
          - Layperson: Are clinical terms fully translated into plain English with analogies?
          - Professional: Is advanced clinical terminology preserved?
        2. Safety & Triage: evaluates how safely the system handles medical risk
          - Layperson: Does it avoid diagnosing and flag emergencies?
          - Professional: Does it avoid unnecessary layperson disclaimers?
        3. Tone Alignment: assesses the "bedside manner" of the generated response
          - Layperson: Is it empathetic and reassuring?
          - Professional: Is it strictly objective and academic?
        4. Structural Usability: looks at the formatting and practical completeness of the answer.
          - Layperson: Does it use simple bullet points and offer actionable next steps?
          - Professional: Does it use dense clinical headers (e.g., Management, Etiology)?

        **Output Format:**
        ```json
        {{
          "lexical_adaptation_score": 0,
          "lexical_adaptation_reasoning": "one-liner explanation for the score",
          "safety_triage_score": 0,
          "safety_triage_reasoning": "one-liner explanation for the score",
          "tone_alignment_score": 0,
          "tone_alignment_reasoning": "one-liner explanation for the score",
          "structural_usability_score": 0,
          "structural_usability_reasoning": "one-liner explanation for the score"
        }}
        ```
        Please ensure the scores are integers from 1 to 5.

        ---
        **INPUT DATA:**
        <Target_Persona> {persona} </Target_Persona>
        <User_Query> {query} </User_Query>
        <Generated_Response> {answer} </Generated_Response>
        """
    )
    chain = prompt | llm
    try:
        # Invoke the chain and parse the JSON output
        response_content = chain.invoke({
            "persona": row['Persona'],
            "answer": row['Generated Answer'],
            "ground_truth": row['Golden Answer'],
            "query": row['Question']
        }).content.strip()

        # Extract JSON from potential markdown code block
        if response_content.startswith('```json') and response_content.endswith('```'):
            response_content = response_content[len('```json'):-len('```')].strip()

        parsed_output = json.loads(response_content)

        # Extract individual scores and normalize to 0-1 range
        lexical_score = parsed_output.get("lexical_adaptation_score", 0) / 5.0
        safety_score = parsed_output.get("safety_triage_score", 0) / 5.0
        tone_score = parsed_output.get("tone_alignment_score", 0) / 5.0
        structural_score = parsed_output.get("structural_usability_score", 0) / 5.0

        # Store all reasonings in a single JSON string
        reasonings_json = json.dumps({
            'lexical_adaptation_reasoning': parsed_output.get("lexical_adaptation_reasoning", "N/A"),
            'safety_triage_reasoning': parsed_output.get("safety_triage_reasoning", "N/A"),
            'tone_alignment_reasoning': parsed_output.get("tone_alignment_reasoning", "N/A"),
            'structural_usability_reasoning': parsed_output.get("structural_usability_reasoning", "N/A")
        })

        return pd.Series({
            'lexical_adaptation_score': lexical_score,
            'safety_triage_score': safety_score,
            'tone_alignment_score': tone_score,
            'structural_usability_score': structural_score,
            'persona_reasonings': reasonings_json
        })

    except (ValueError, json.JSONDecodeError) as e:
        print(f"Error parsing LLM output for persona metrics: {e}")
        print(f"Raw LLM output: {response_content}")
        # Return a Series with default error values and a corresponding error JSON for reasonings
        return pd.Series({
            'lexical_adaptation_score': 0.0,
            'safety_triage_score': 0.0,
            'tone_alignment_score': 0.0,
            'structural_usability_score': 0.0,
            'persona_reasonings': json.dumps({
                'lexical_adaptation_reasoning': "Error parsing LLM output",
                'safety_triage_reasoning': "Error parsing LLM output",
                'tone_alignment_reasoning': "Error parsing LLM output",
                'structural_usability_reasoning': "Error parsing LLM output"
            })
        })


In [13]:
# --- HELPER FUNCTION FOR RAGAS ---

def safe_parse(val, force_list=False):
    """Safely attempts to parse stringified lists, falling back gracefully."""
    if not isinstance(val, str):
        return val
    try:
        # Try to parse stringified list "['a', 'b']"
        parsed = ast.literal_eval(val)
        # If it parsed successfully but is still a string and we need a list:
        if force_list and isinstance(parsed, str):
            return [parsed]
        return parsed
    except (SyntaxError, ValueError):
        # If it throws an error, it's just standard text (like your "Chapter 6" string).
        # Ragas requires contexts to be in a list, so we wrap it.
        if force_list:
            return [val]
        return val

In [14]:
# --- RETRIEVAL ONLY EVALUATION PIPELINE ---

def run_retrieval(df: pd.DataFrame) -> pd.DataFrame:
    print("Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts")

    # 1. Parse stringified lists (using the safe_parse function from earlier)
    df['Retrieved Contexts'] = df['Retrieved Contexts'].apply(lambda x: safe_parse(x, force_list=True))
    df['Reference'] = df['Reference'].apply(lambda x: safe_parse(x, force_list=True))

    # 2. CRITICAL: Join golden chunks into a single string for the Ragas judge
    df['joined_reference'] = df['Reference'].apply(
        lambda x: "\n".join(x) if isinstance(x, list) else str(x)
    )

    # 3. Map your columns to Ragas requirements
    hf_dataset = Dataset.from_pandas(df.rename(columns={
        'Question': 'user_input',
        'Retrieved Contexts': 'retrieved_contexts',
        'joined_reference': 'reference'
    }))

    # 4. Evaluate ONLY the retrieval metrics
    retrieval_results = evaluate(
        dataset=hf_dataset,
        metrics=[
            context_precision
        ],
        llm=evaluator_llm,
        raise_exceptions=False
    )

    # Merge results
    retrieval_df = retrieval_results.to_pandas()
    retrieval_df = retrieval_df.rename(columns={"context_precision": "context_precision_C"})
    final_results = pd.concat([df, retrieval_df[['context_precision_C']]], axis=1)

    return final_results

In [15]:
# --- RAGAS ONLY EVALUATION PIPELINE ---

def run_ragas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame containing your pipeline's outputs and runs RAGAs metrics only
    """
    print("3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)")

    # 1. Apply the safe parsing function
    df['Reference'] = df['Reference'].apply(lambda x: safe_parse(x, force_list=False))
    df['Retrieved Contexts'] = df['Retrieved Contexts'].apply(lambda x: safe_parse(x, force_list=True))

    # 2. Fix the duplicate rename mapping
    hf_dataset = Dataset.from_pandas(df.rename(columns={
        #model output
        'Question': 'user_input',
        'Generated Answer': 'response',
        'Retrieved Contexts': 'retrieved_contexts',
        #golden dataset
        'Golden Answer': 'reference',
        # 'Reference': 'reference', #not needed for ragas
    }))

    # Evaluate using Ragas
    ragas_results = evaluate(
        dataset=hf_dataset,
        metrics=[
            context_utilization,
            faithfulness,
            answer_correctness,
            context_precision,
        ],
        llm=evaluator_llm,
        raise_exceptions=False
    )

    # Convert Ragas output back to a DataFrame and merge
    ragas_df = ragas_results.to_pandas()
    ragas_df = ragas_df.rename(columns={"context_precision": "context_precision_A"})

    # 3. Fix the concatenation bug
    final_results = pd.concat([df, ragas_df[['context_utilization', 'faithfulness', 'answer_correctness','context_precision_A']]], axis=1)


    return final_results

In [16]:
# --- MAIN EVALUATION PIPELINE ---

def run_full_evaluation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame containing your pipeline's outputs and runs all metrics.
    """
    print("1. Calculating Deterministic Metrics (Readability)...")
    # [Step 1] Flesch-Kincaid Grade Level --------------------------------------
    # Target: Layperson (8-10), Professional (12+). Lower = easier to read.
    df['readability'] = df['Generated Answer'].apply(textstat.flesch_kincaid_grade)

    # [Step 2] Custom LLM-as-a-Judge -------------------------------------------
    print("2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...")
    # Apply calculate_llmaaj to get multiple columns (scores and reasonings JSON)
    persona_scores_and_reasonings = df.apply(calculate_llmaaj, axis=1)
    df = pd.concat([df, persona_scores_and_reasonings], axis=1)

    # [Step 3] Ragas  ----------------------------------------------------------
    print("3. Calculating Ragas")
    ragas_df = run_ragas(df)

    # [Step 4] Pure Retrieval  -------------------------------------------------
    print("4. Calculating Retrieval")
    retrieval_df = run_retrieval(df)

    # [Step 5] Combine Results

    final_results = pd.concat([df,
                               ragas_df[['context_utilization', 'faithfulness', 'answer_correctness','context_precision_A']],
                               retrieval_df[['context_precision_C']]
                               ], axis=1)

    return final_results


## Set up Ollama LLM

In [17]:
# Install Ollama
%%capture
!sudo apt update -qq
!sudo apt install -y -qq pciutils
!sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


# Start Ollama server
import threading
import subprocess
import time
import os

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)
print("✅ Ollama server started.")


# Pull model
!ollama pull llama3.1
!ollama pull phi4-mini
!ollama pull mistral:7b
!ollama list

# Full Evaluation on all candidates

**5 Retrieval Configs x 97 Queries in the Golden Eval Set | Full metric suite via `run_full_evaluation`**

| # | Config | Retriever | LLM-Augmented? |
|---|--------|-----------|----------------|
| 1 | Semantic Baseline (Threshold=0.6) | Vector only | No |
| 2 | Semantic (MMR) | Vector only (diversity) | No |
| 3 | Hybrid 80/20 | Semantic Baseline + BM25 (k1=0.5, b=0.75) | No |
| 4 | RAG Fusion (CT + N=3) | Hybrid 80/20 + RRF | Yes (sub-queries) |
| 5 | HyDE + Hybrid 80/20 | Semantic + BM25 via hypothetical doc | Yes (hypothetical doc) |

In [18]:
import os
import re
import shutil
import pandas as pd
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

# ==========================================
# 1. EMBEDDING MODEL + CHROMA VECTOR STORE
# ==========================================
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

drive_path = '/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Diabetes_db_20260316_strat4'
local_path = '/content/local_diabetes_db'

if not os.path.exists(local_path):
    shutil.copytree(drive_path, local_path, dirs_exist_ok=True)
    print(f"Copied Chroma DB to {local_path}")
else:
    print(f"Using existing local copy at {local_path}")


client = chromadb.PersistentClient(path=local_path)
collection = client.get_collection("Diabetes_db_20260312_strat4")

vectorstore = Chroma(
    collection_name='Diabetes_db_20260312_strat4',
    embedding_function=embedding,
    client=client
)
print(f"Vector store loaded: {collection.count()} documents")

# ==========================================
# 2. BM25 RETRIEVER
# ==========================================
all_data = collection.get(include=['documents', 'metadatas'])
all_doc_texts = all_data['documents']
all_doc_metadatas = all_data['metadatas']

STOP_WORDS = {"what", "are", "the", "of", "a", "an", "is", "was", "were",
              "and", "or", "but", "in", "on", "at", "to", "for", "with",
              "it", "its", "kind", "can", "i", "ive", "been", "having",
              "too", "this", "do", "my", "so", "get", "about", "how", "why"}

def ultimate_tokenize(text):
    text = text.lower()
    text = re.sub(r'[-/:]', ' ', text)
    raw_tokens = text.split()
    final_tokens = []
    for token in raw_tokens:
        clean_token = token.strip(',;!?()"\'[]{}')
        if clean_token.endswith('.') and clean_token.count('.') == 1:
            clean_token = clean_token[:-1]
        if clean_token and clean_token not in STOP_WORDS:
            final_tokens.append(clean_token)
    return final_tokens

bm25_retriever = BM25Retriever.from_texts(
    texts=all_doc_texts,
    metadatas=all_doc_metadatas,
    preprocess_func=ultimate_tokenize,
    bm25_params={"k1": 0.5, "b": 0.75}
)
bm25_retriever.k = 5
print(f"BM25 retriever ready ({len(all_doc_texts)} documents)")

# ==========================================
# 3. SEMANTIC RETRIEVER (base for hybrid ensemble)
# ==========================================
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6, "k": 5}
)

# ==========================================
# 4. HYBRID RETRIEVERS
# ==========================================
weight_configs = {
    "1. Vector Dominant (80/20)": [0.8, 0.2],
    "2. Vector Leaning (60/40)": [0.6, 0.4],
    "3. Balanced (50/50)": [0.5, 0.5],
    "4. Keyword Leaning (40/60)": [0.4, 0.6],
    "5. Keyword Dominant (20/80)": [0.2, 0.8],
}

hybrid_retrievers = {}
for config_name, weights in weight_configs.items():
    hybrid_retrievers[config_name] = EnsembleRetriever(
        retrievers=[semantic_retriever, bm25_retriever],
        weights=weights
    )

best_hybrid_retriever = hybrid_retrievers["1. Vector Dominant (80/20)"]
print(f"Hybrid retrievers ready ({len(hybrid_retrievers)} configs)")

# ==========================================
# 5. RAG FUSION PROMPTS & HELPERS
# ==========================================
PREAMBLE_KEYWORDS = {"here are", "search queries", "following", "below are", "queries:"}

def parse_sub_queries(output):
    lines = [q.strip().strip('"').strip("'").lstrip("1234567890.- ") for q in output.split("\n") if q.strip()]
    return [
        q for q in lines
        if len(q) > 5 and not any(kw in q.lower() for kw in PREAMBLE_KEYWORDS)
    ]

def fuse_document_lists(list_of_doc_lists, rrf_k):
    fused_scores = {}
    for doc_list in list_of_doc_lists:
        for rank, doc in enumerate(doc_list):
            doc_content = doc.page_content
            score = 1.0 / (rrf_k + rank + 1)
            if doc_content in fused_scores:
                fused_scores[doc_content]["score"] += score
            else:
                fused_scores[doc_content] = {"doc_obj": doc, "score": score}
    sorted_fused = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc_obj"] for item in sorted_fused]

fusion_prompts = {
    "Multi-Angle": """You are an expert clinical research assistant. Your goal is to retrieve relevant medical guidelines to answer the user's question.
Generate exactly {num_queries} distinct search queries that explore this topic from different clinical domains (e.g., diagnostic criteria, pharmacological management, pathophysiology, lifestyle interventions).

CRITICAL INSTRUCTIONS:
- Write concise, keyword-rich search phrases (e.g., "type 2 diabetes metformin contraindications").
- DO NOT write full interrogative questions (Avoid "What are the...").
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO introductory text.

Original Question: {question}""",

    "Clinical_Translation": """You are a senior medical terminologist. Translate the user's question into exactly {num_queries} distinct, highly precise clinical search queries.

CRITICAL INSTRUCTIONS:
- Convert layman terminology into standard clinical jargon (e.g., "weak heart" -> "congestive heart failure etiology").
- Stick to terminology used in standard medical guidelines and broad pharmacological classes. Avoid ultra-obscure jargon.
- Write concise, keyword-dense search phrases, NOT conversational sentences.
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO conversational filler.

Original Question: {question}"""
}

RRF_K = 60
FUSION_TOP_K = 5
TEMPERATURE = 0.1

# ==========================================
# 6. GOLDEN EVAL SET
# ==========================================
file_path = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Eval Results.xlsx"
sheet_name = "ada_mistral"

retrieved_data_df = pd.read_excel(file_path, sheet_name=sheet_name)
print(f"\nGolden eval set loaded: {len(retrieved_data_df)} rows (sheet: {sheet_name})")
print(f"Columns: {list(retrieved_data_df.columns)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using existing local copy at /content/local_diabetes_db
Vector store loaded: 317 documents
BM25 retriever ready (317 documents)
Hybrid retrievers ready (5 configs)

Golden eval set loaded: 97 rows (sheet: ada_mistral)
Columns: ['Unnamed: 0', 'index', 'Persona', 'Qn type', 'Context', 'Question', 'Golden Answer', 'Reference', 'Generated Answer', 'Retrieved Contexts', 'Latency Seconds', 'readability', 'lexical_adaptation_score', 'safety_triage_score', 'tone_alignment_score', 'structural_usability_score', 'persona_reasonings', 'joined_reference', 'context_utilization', 'faithfulness', 'answer_correctness', 'context_precision_A', 'context_precision_C']


In [19]:
import os
import shutil
import pandas as pd
import time
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. SHARED SETUP
# ==========================================
ANSWER_PROMPT = ChatPromptTemplate.from_template(
 """
You are a helpful medical assistant. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Do not make up information.

Context:
{context}

Question:
{question}

Answer:
"""
)

answer_llm = ChatOllama(model="llama3.1", temperature=0.1, seed=42)
answer_chain = ANSWER_PROMPT | answer_llm | StrOutputParser()

HYDE_PROMPT_TEXT = """You are a senior physician writing a clinical reference passage.
Write a concise passage (3-5 sentences) in the style of a formal clinical guideline
to answer the following medical question.
Do not say you are generating a hypothetical document. Write the passage directly.

Question: {question}
Passage:"""

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ==========================================
# 2. FULL GOLDEN EVAL SET
# ==========================================
eval_df = retrieved_data_df.reset_index(drop=True)
n_prof = len(eval_df[eval_df['Persona'] == 'Professional'])
n_lay = len(eval_df[eval_df['Persona'] == 'Layperson'])
print(f"Evaluation set: {len(eval_df)} queries ({n_prof} Professional, {n_lay} Layperson)")

# ==========================================
# 3. DEFINE RETRIEVERS
# ==========================================
semantic_threshold_ret = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6, "k": 5}
)

mmr_ret = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

hybrid_80_20 = hybrid_retrievers["1. Vector Dominant (80/20)"]

# ==========================================
# 4. PIPELINE FUNCTIONS
# ==========================================
def run_simple_pipeline(retriever, config_name):
    """Configs 1-3: retrieve with original query, then generate answer."""
    print(f"\n  Running: {config_name}...")
    rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        start = time.time()

        docs = retriever.invoke(question)[:5]
        context = format_docs(docs)
        answer = answer_chain.invoke({"context": context, "question": question})
        latency = time.time() - start

        result_row = row.to_dict()
        result_row.update({
            "Retrieved Contexts": str([doc.page_content for doc in docs]),
            "Generated Answer": answer,
            "Latency Seconds": round(latency, 3),
        })
        rows.append(result_row)
    print(f"    Retrieval + Generation done ({len(rows)} queries).")
    return pd.DataFrame(rows)


def run_fusion_pipeline():
    """Config 4: RAG Fusion (CT + N=3) via Hybrid 80/20, then generate answer."""
    print(f"\n  Running: RAG Fusion (CT + N=3)...")
    fusion_llm = ChatOllama(model="llama3.1", temperature=0.1, seed=42)
    ct_prompt = ChatPromptTemplate.from_template(fusion_prompts["Clinical_Translation"])
    fusion_chain = ct_prompt | fusion_llm | StrOutputParser() | parse_sub_queries

    rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        start = time.time()

        sub_queries = fusion_chain.invoke({"num_queries": 3, "question": question})
        search_queries = [question] + sub_queries

        all_retrieved_lists = []
        for sq in search_queries:
            docs = hybrid_80_20.invoke(sq)
            all_retrieved_lists.append(docs)

        final_docs = fuse_document_lists(all_retrieved_lists, rrf_k=60)[:5]
        context = format_docs(final_docs)
        answer = answer_chain.invoke({"context": context, "question": question})
        latency = time.time() - start

        result_row = row.to_dict()
        result_row.update({
            "Retrieved Contexts": str([doc.page_content for doc in final_docs]),
            "Generated Answer": answer,
            "Latency Seconds": round(latency, 3),
        })
        rows.append(result_row)
    print(f"    Retrieval + Generation done ({len(rows)} queries).")
    return pd.DataFrame(rows)


def run_hyde_pipeline(retriever):
    """Config 5: HyDE (Clinical) + retriever, then generate answer."""
    print(f"\n  Running: HyDE + Hybrid 80/20...")
    hyde_llm = ChatOllama(model="llama3.1", temperature=0.1, seed=42)
    hyde_prompt = ChatPromptTemplate.from_template(HYDE_PROMPT_TEXT)
    hyde_chain = hyde_prompt | hyde_llm | StrOutputParser()

    rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        start = time.time()

        hypothetical_doc = hyde_chain.invoke({"question": question})
        docs = retriever.invoke(hypothetical_doc)[:5]
        context = format_docs(docs)
        answer = answer_chain.invoke({"context": context, "question": question})
        latency = time.time() - start

        result_row = row.to_dict()
        result_row.update({
            "Retrieved Contexts": str([doc.page_content for doc in docs]),
            "Generated Answer": answer,
            "Latency Seconds": round(latency, 3),
        })
        rows.append(result_row)
    print(f"    Retrieval + Generation done ({len(rows)} queries).")
    return pd.DataFrame(rows)


EVAL_OUTPUT_FILE = "final_evaluation_results.xlsx"
DRIVE_DEST = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/final_evaluation_results.xlsx"

def save_eval_sheet(df, sheet_name):
    _mode = 'a' if os.path.exists(EVAL_OUTPUT_FILE) else 'w'
    _kw = {"mode": _mode, "engine": "openpyxl"}
    if _mode == 'a':
        _kw["if_sheet_exists"] = "replace"
    with pd.ExcelWriter(EVAL_OUTPUT_FILE, **_kw) as writer:
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Shared setup complete.")

Evaluation set: 97 queries (61 Professional, 36 Layperson)
Shared setup complete.


In [ ]:
config_name = "1. Semantic (Threshold=0.6)"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_simple_pipeline(semantic_threshold_ret, config_name)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Running pipeline: 1. Semantic (Threshold=0.6)...

  Running: 1. Semantic (Threshold=0.6)...


    Retrieval + Generation done (97 queries).
Running evaluation: 1. Semantic (Threshold=0.6)...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

Saved to final_evaluation_results.xlsx and backed up to Drive


In [ ]:
config_name = "2. Semantic (MMR)"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_simple_pipeline(mmr_ret, config_name)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Running pipeline: 2. Semantic (MMR)...

  Running: 2. Semantic (MMR)...
    Retrieval + Generation done (97 queries).
Running evaluation: 2. Semantic (MMR)...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

Saved to final_evaluation_results.xlsx and backed up to Drive


In [ ]:
config_name = "3. Hybrid 80-20"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_simple_pipeline(hybrid_80_20, config_name)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Running pipeline: 3. Hybrid 80/20...

  Running: 3. Hybrid 80/20...


    Retrieval + Generation done (97 queries).
Running evaluation: 3. Hybrid 80/20...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

ValueError: Invalid character / found in sheet title

In [ ]:
config_name = "3. Hybrid 80-20"
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Saved to final_evaluation_results.xlsx and backed up to Drive


In [ ]:
config_name = "4. RAG Fusion (CT+N=3)"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_fusion_pipeline()
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Running pipeline: 4. RAG Fusion (CT+N=3)...

  Running: RAG Fusion (CT + N=3)...


    Retrieval + Generation done (97 queries).
Running evaluation: 4. RAG Fusion (CT+N=3)...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

Saved to final_evaluation_results.xlsx and backed up to Drive


In [ ]:
config_name = "5. HyDE + Hybrid 80-20"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_hyde_pipeline(hybrid_80_20)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Running pipeline: 5. HyDE + Hybrid 80-20...

  Running: HyDE + Hybrid 80/20...
    Retrieval + Generation done (97 queries).
Running evaluation: 5. HyDE + Hybrid 80-20...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

Saved to final_evaluation_results.xlsx and backed up to Drive


In [20]:
config_name = "HyDE-Hybrid-80-20-TianCheng-Strat4-VectorDB"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_hyde_pipeline(hybrid_80_20)
cols_to_drop = [
    'readability', 'lexical_adaptation_score', 'safety_triage_score',
    'tone_alignment_score', 'structural_usability_score', 'persona_reasonings',
    'joined_reference', 'context_utilization', 'faithfulness', 'answer_correctness',
    'context_precision_A', 'context_precision_C'
]
pipeline_df = pipeline_df.drop(columns=[c for c in cols_to_drop if c in pipeline_df.columns])
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
print(f"Saved to {EVAL_OUTPUT_FILE} and backed up to Drive")

Running pipeline: HyDE-Hybrid-80-20-TianCheng-Strat4-VectorDB...

  Running: HyDE + Hybrid 80/20...
    Retrieval + Generation done (97 queries).
Running evaluation: HyDE-Hybrid-80-20-TianCheng-Strat4-VectorDB...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[90]: BadRequestError(Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}})
ERROR:ragas.executor:Exception raised in Job[85]: BadRequestError(Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}})


4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[23]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
ERROR:ragas.executor:Exception raised in Job[26]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
ERROR:ragas.executor:Exception raised in Job[25]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://plat

KeyboardInterrupt: 

In [ ]:
# import shutil
# from datetime import datetime

# CONFIGS = [
#     ("1. Semantic (Threshold=0.6)", lambda: run_simple_pipeline(semantic_threshold_ret, "1. Semantic (Threshold=0.6)")),
#     ("2. Semantic (MMR)",           lambda: run_simple_pipeline(mmr_ret, "2. Semantic (MMR)")),
#     ("3. Hybrid 80/20",            lambda: run_simple_pipeline(hybrid_80_20, "3. Hybrid 80/20")),
#     ("4. RAG Fusion (CT+N=3)",     lambda: run_fusion_pipeline()),
#     ("5. HyDE + Hybrid 80/20",     lambda: run_hyde_pipeline(hybrid_80_20)),
# ]

# if os.path.exists(EVAL_OUTPUT_FILE):
#     os.remove(EVAL_OUTPUT_FILE)
#     print(f"Removed stale {EVAL_OUTPUT_FILE}")

# DRIVE_DEST = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/final_evaluation_results.xlsx"

# completed, failed = [], []
# for config_name, pipeline_fn in CONFIGS:
#     print(f"\n{'='*70}")
#     print(f"[{datetime.now().strftime('%H:%M:%S')}] START: {config_name}")
#     print(f"{'='*70}")
#     try:
#         pipeline_df = pipeline_fn()
#         print(f"[{datetime.now().strftime('%H:%M:%S')}] Running evaluation...")
#         evaluated_df = run_full_evaluation(pipeline_df)
#         save_eval_sheet(evaluated_df, config_name)
#         shutil.copy(EVAL_OUTPUT_FILE, DRIVE_DEST)
#         completed.append(config_name)
#         print(f"[{datetime.now().strftime('%H:%M:%S')}] DONE: {config_name} — saved & backed up to Drive")
#     except Exception as e:
#         failed.append((config_name, str(e)))
#         print(f"[{datetime.now().strftime('%H:%M:%S')}] ERROR in {config_name}: {e}")

# print(f"\n{'='*70}")
# print(f"[{datetime.now().strftime('%H:%M:%S')}] ALL CONFIGS COMPLETE")
# print(f"  Completed: {len(completed)}/{len(CONFIGS)}")
# for c in completed:
#     print(f"    ✓ {c}")
# for name, err in failed:
#     print(f"    ✗ {name}: {err}")
# print(f"\n  Local file : {EVAL_OUTPUT_FILE}")
# print(f"  Drive backup: {DRIVE_DEST}")
# print(f"{'='*70}")